In [33]:
import json

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, AdamW

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 一 参数

In [59]:
class Arguments(object):
    bert_dir = '/Users/bowie/Documents/muti-model/bert-base-chinese'

    # subject 的 predicate 是 object
    train_path = 'train.json'
    dev_path = 'dev.json'
    test_path = 'test.json'
    rel_path = 'rel.json'

    with open(rel_path, 'r', encoding='utf-8') as f:
        rel_data = json.load(f)
    
    id2rel = {int(i): rel for i, rel in rel_data.items()}
    print(id2rel)
    rel2id = {rel: int(i) for i, rel in rel_data.items()}
    print(rel2id)

    # 建议将 max_length 设置为略大于文本的最大长度，如 300。这样可以避免不必要的截断，同时保持计算效率和性能的平衡。
    max_len = 300

args = Arguments()


{0: '出品公司', 1: '国籍', 2: '出生地', 3: '民族', 4: '出生日期', 5: '毕业院校', 6: '歌手', 7: '所属专辑', 8: '作词', 9: '作曲', 10: '连载网站', 11: '作者', 12: '出版社', 13: '主演', 14: '导演', 15: '编剧', 16: '上映时间', 17: '成立日期'}
{'出品公司': 0, '国籍': 1, '出生地': 2, '民族': 3, '出生日期': 4, '毕业院校': 5, '歌手': 6, '所属专辑': 7, '作词': 8, '作曲': 9, '连载网站': 10, '作者': 11, '出版社': 12, '主演': 13, '导演': 14, '编剧': 15, '上映时间': 16, '成立日期': 17}


In [54]:
tokenizer = BertTokenizer.from_pretrained(args.bert_dir)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [57]:
# 找文本的最大长度
for i in [args.train_path, args.dev_path, args.test_path]:
    max_length = 0
    max_text = ''
    min_length = 300
    min_text = ''
    with open(i, 'r', encoding='utf-8') as f:
        for i in f:
            data_line = json.loads(i)
            data_token = tokenizer.tokenize(data_line['text'])
            if len(data_token) > max_length:
                max_length = len(data_token)
                max_text = data_token
            if len(data_token) < min_length:
                min_length = len(data_token)
                min_text = data_token
    print(max_length)
    print(max_text)
    print(min_length)
    print(min_text)
    print('------------------------')

294
['吴', '宇', '森', '的', '电', '影', '可', '以', '这', '样', '分', '为', '几', '个', '系', '列', '，', '早', '期', '的', '《', '帝', '女', '花', '》', '、', '《', '钱', '作', '怪', '》', '以', '致', '《', '英', '雄', '无', '泪', '》', '、', '《', '笑', '匠', '》', '等', '可', '以', '看', '做', '吴', '宇', '森', '初', '期', '练', '手', '之', '作', '，', '从', '吴', '宇', '森', '的', '早', '期', '电', '影', '可', '以', '看', '出', '他', '不', '是', '那', '种', '很', '有', '天', '赋', '的', '人', '物', '，', '虽', '然', '有', '些', '才', '华', '但', '并', '不', '是', '特', '别', '出', '色', '，', '但', '吴', '宇', '森', '在', '拍', '摄', '了', '这', '近', '十', '部', '影', '片', '之', '后', '，', '手', '法', '开', '始', '纯', '属', '起', '来', '，', '一', '直', '到', '接', '拍', '了', '《', '英', '雄', '本', '色', '》', '，', '其', '能', '力', '才', '让', '人', '一', '目', '了', '然', '，', '其', '电', '影', '风', '格', '特', '色', '开', '始', '成', '熟', '起', '来', '，', '这', '也', '开', '启', '了', '吴', '宇', '森', '江', '湖', '片', '的', '时', '代', '，', '《', '英', '雄', '本', '色', '2', '》', '、', '《', '义', '胆', '群', '英', '》', '、', '《', '喋', '血', '双', '雄', 

## 二 数据处理

In [ ]:
# 自定义数据集类
class SPODataset(Dataset):
    def __init__(self, dataset_path):
        self.data = []
        with open(dataset_path, 'r', encoding='utf-8') as f:
            for i in f:
                data_line = json.loads(i)
                self.data.append(data_line)

    def __len__(self):
        # 返回数据集的大小
        return len(self.data)

    def __getitem__(self, idx):
        # 根据索引返回一个样本及其对应的标签
        content = self.data[idx]
        text = content['text']
        spo_list = content['spo_list']

        # truncation=True：对输入序列中的所有部分（单句或句子对）进行截断。
        # truncation='only_first'：只截断第一个句子，第二个句子不受影响。
        # tokens = tokenizer.tokenize(text,max_length=args.max_len, padding='max_length', truncation=True, return_tensors='pt')
        tokens = tokenizer.tokenize(text)

        def find_head_idx(source, target):
            target_len = len(target)
            for i in range(len(source)):
                if source[i: i + target_len] == target:
                    return i
            return -1
        
        s2ro_map = {}
        for triple in spo_list:
            triple = (tokenizer.tokenize(triple['subject']), triple['predicate'], tokenizer.tokenize(triple['object']))
            sub_head_idx = find_head_idx(tokens, triple[0])
            obj_head_idx = find_head_idx(tokens, triple[2])
            if sub_head_idx != -1 and obj_head_idx != -1:
                sub = (sub_head_idx, sub_head_idx + len(triple[0]) - 1)
                if sub not in s2ro_map:
                    s2ro_map[sub] = []
                s2ro_map[sub].append((obj_head_idx,
                                    obj_head_idx + len(triple[2]) - 1,
                                    args.rel2id[triple[1]]))
                
        print(s2ro_map)
        return text, spo_list


In [60]:
test_text = [
    {"text": "1997年，李柏光从北京大学法律系博士毕业", 
     "spo_list": [{"predicate": "毕业院校", "object_type": "学校", "subject_type": "人物", "object": "北京大学", "subject": "李柏光"}]}
]

In [72]:
tokens_1 = tokenizer(test_text[0]['text'], max_length=30, padding='max_length', truncation=True)

In [73]:
tokens_1

{'input_ids': [101, 8387, 2399, 8024, 3330, 3377, 1045, 794, 1266, 776, 1920, 2110, 3791, 2526, 5143, 1300, 1894, 3684, 689, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [63]:
text = test_text[0]['text']
spo_list = test_text[0]['spo_list']

# truncation=True：对输入序列中的所有部分（单句或句子对）进行截断。
# truncation='only_first'：只截断第一个句子，第二个句子不受影响。
# tokens = tokenizer.tokenize(text,max_length=args.max_len, padding='max_length', truncation=True, return_tensors='pt')
tokens = tokenizer.tokenize(text)
print(tokens)

def find_head_idx(source, target):
    target_len = len(target)
    for i in range(len(source)):
        if source[i: i + target_len] == target:
            return i
    return -1

s2ro_map = {}
for triple in spo_list:
    triple = (tokenizer.tokenize(triple['subject']), triple['predicate'], tokenizer.tokenize(triple['object']))
    sub_head_idx = find_head_idx(tokens, triple[0])
    obj_head_idx = find_head_idx(tokens, triple[2])
    if sub_head_idx != -1 and obj_head_idx != -1:
        sub = (sub_head_idx, sub_head_idx + len(triple[0]) - 1)
        if sub not in s2ro_map:
            s2ro_map[sub] = []
        s2ro_map[sub].append((obj_head_idx,
                            obj_head_idx + len(triple[2]) - 1,
                            args.rel2id[triple[1]]))
        
print(s2ro_map)

['1997', '年', '，', '李', '柏', '光', '从', '北', '京', '大', '学', '法', '律', '系', '博', '士', '毕', '业']
{(3, 5): [(7, 10, 5)]}


In [ ]:
token_ids = torch.tensor(tokens, dtype=torch.long)
masks = torch.tensor(masks, dtype=torch.bool)
sub_heads, sub_tails = torch.zeros(text_len), torch.zeros(text_len)
sub_head, sub_tail = torch.zeros(text_len), torch.zeros(text_len)
obj_heads = torch.zeros((text_len, self.config.num_relations))
obj_tails = torch.zeros((text_len, self.config.num_relations))

In [ ]:
if s2ro_map:
    for s in s2ro_map:
        sub_heads[s[0]] = 1
        sub_tails[s[1]] = 1
    sub_head_idx, sub_tail_idx = choice(list(s2ro_map.keys()))
    sub_head[sub_head_idx] = 1
    sub_tail[sub_tail_idx] = 1
    for ro in s2ro_map.get((sub_head_idx, sub_tail_idx), []):
        obj_heads[ro[0]][ro[2]] = 1
        obj_tails[ro[1]][ro[2]] = 1

In [65]:
tokenizer.encode("1997年，李柏光从北京大学法律系博士毕业")

[101,
 8387,
 2399,
 8024,
 3330,
 3377,
 1045,
 794,
 1266,
 776,
 1920,
 2110,
 3791,
 2526,
 5143,
 1300,
 1894,
 3684,
 689,
 102]

In [29]:
tokens_batch, segments_batch, sub_heads_batch, sub_tails_batch, sub_head_batch, sub_tail_batch, obj_heads_batch, obj_tails_batch = [], [], [], [], [], [], [], []


if s2ro_map:
    token_ids, segment_ids = tokenizer.encode(first=text)
    if len(token_ids) > text_len:
        token_ids = token_ids[:text_len]
        segment_ids = segment_ids[:text_len]
    tokens_batch.append(token_ids)
    segments_batch.append(segment_ids)
    sub_heads, sub_tails = np.zeros(text_len), np.zeros(text_len)
    for s in s2ro_map:
        sub_heads[s[0]] = 1     
        sub_tails[s[1]] = 1     
    sub_head, sub_tail = choice(list(s2ro_map.keys()))
    obj_heads, obj_tails = np.zeros((text_len, self.num_rels)), np.zeros((text_len, self.num_rels))
    for ro in s2ro_map.get((sub_head, sub_tail), []): 
        obj_heads[ro[0]][ro[2]] = 1
        obj_tails[ro[1]][ro[2]] = 1
    sub_heads_batch.append(sub_heads)
    sub_tails_batch.append(sub_tails)
    sub_head_batch.append([sub_head])
    sub_tail_batch.append([sub_tail])
    obj_heads_batch.append(obj_heads)
    obj_tails_batch.append(obj_tails)
    if len(tokens_batch) == self.batch_size or idx == idxs[-1]:
        tokens_batch = seq_padding(tokens_batch)
        segments_batch = seq_padding(segments_batch)
        sub_heads_batch = seq_padding(sub_heads_batch)
        sub_tails_batch = seq_padding(sub_tails_batch)
        obj_heads_batch = seq_padding(obj_heads_batch, np.zeros(self.num_rels))
        obj_tails_batch = seq_padding(obj_tails_batch, np.zeros(self.num_rels))
        sub_head_batch, sub_tail_batch = np.array(sub_head_batch), np.array(sub_tail_batch)


In [30]:
load_data(args.train_path)

train_data len: 55959 [{'text': '《亲子系列丛书（全3册）》是2002年广西人民出版社出版的图书，作者是卢美贵', 'spo_list': [('亲子系列丛书（全3册）', '作者', '卢美贵'), ('亲子系列丛书（全3册）', '出版社', '广西人民出版社')]}, {'text': '《新刑事诉讼法案例解读4》由最高人民法院张军副院长和中国人民大学法学院陈卫东教授共同主编', 'spo_list': [('陈卫东', '国籍', '中国')]}]


In [34]:
text = "1997年，李柏光从北京大学法律系博士毕业"

In [43]:
encoding = tokenizer(text, padding="max_length", truncation=True, max_length=30)

In [44]:
# encoding = tokenizer(text, padding=True, return_offsets_mapping=True, return_tensors="pt")
input_ids = encoding['input_ids']  # tokens的ID表示
attention_mask = encoding['attention_mask']  # attention mask
token_type_ids = encoding['token_type_ids']  # segment IDs
# offsets = encoding['offset_mapping']  # token在原始句子中的位置

In [ ]:
subject = "李柏光"
subject_start = text.find(subject)
subject_end = subject_start + len(subject)

# 查找subject在tokens中的head和tail位置
sub_heads = [0] * len(input_ids[0])
sub_tails = [0] * len(input_ids[0])

for i, (start, end) in enumerate(offsets[0]):
    if start == subject_start:
        sub_heads[i] = 1  # 标记head
    if end == subject_end:
        sub_tails[i] = 1  # 标记tail


In [ ]:
obj = "北京大学"
obj_start = text.find(obj)
obj_end = obj_start + len(obj)

obj_heads = [0] * len(input_ids[0])
obj_tails = [0] * len(input_ids[0])

for i, (start, end) in enumerate(offsets[0]):
    if start == obj_start:
        obj_heads[i] = 1  # 标记object的head
    if end == obj_end:
        obj_tails[i] = 1  # 标记object的tail
